# Project 8 — Module 9: Fundamentos de Big Data
## Lesson 05 — Evaluation

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 05 — Evaluation |
| **Module** | M9 — Fundamentos de Big Data (Alkemy Bootcamp) |
| **Dataset** | Brazilian E-Commerce (Olist) + Synthetic clickstream |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook evaluates the ML pipeline built in NB 04. It assesses Logistic
> Regression performance (confusion matrix, precision, recall, F1, AUC-ROC),
> interprets K-Means clusters with business profiles, generates marketing
> recommendations per segment, and validates results against the success metrics
> defined in the Problem Statement Canvas (NB 01). The key output is an actionable
> insights report for RetailMax's marketing team.

## Table of Contents

1. [Environment Setup + Data Reload](#1-environment-setup)
2. [Logistic Regression — Evaluation](#2-logistic-regression--evaluation)
3. [K-Means — Cluster Interpretation](#3-k-means--cluster-interpretation)
4. [Cluster Visualization](#4-cluster-visualization)
5. [Success Metrics Validation](#5-success-metrics-validation)
6. [Marketing Insights Report](#6-marketing-insights-report)
7. [LEAN Filter — Waste Elimination Review](#7-lean-filter--waste-elimination-review)
8. [Decisions Log — Lesson 05](#8-decisions-log--lesson-05)
9. [Next Steps — Lesson 06 Preview](#9-next-steps--lesson-06-preview)

---

## 1. Environment Setup + Data Reload <a id='1-environment-setup'></a>

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import sys
import warnings
from pathlib import Path

os.environ['HADOOP_HOME'] = str(Path.home() / 'hadoop')
os.environ['PYSPARK_PYTHON'] = sys.executable

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_FINAL     = Path('../data/final')
REPORTS_FIGURES = Path('../reports/figures')
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

print('Imports ready.')

In [ ]:
# =============================================================================
# SPARK SESSION + LOAD PARQUET FILES FROM NB04
# =============================================================================
spark = (
    SparkSession.builder
    .appName('RetailMax-BigData-Pipeline')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

# Load ML outputs from NB04
lr_predictions = spark.read.parquet(str(DATA_FINAL / 'lr_predictions.parquet'))
km_clusters    = spark.read.parquet(str(DATA_FINAL / 'km_clusters.parquet'))
cluster_profiles = spark.read.parquet(str(DATA_FINAL / 'cluster_profiles.parquet'))

print(f'Spark {spark.version}')
print(f'LR predictions : {lr_predictions.count():,} rows')
print(f'KM clusters    : {km_clusters.count():,} rows')
print(f'Cluster profiles: {cluster_profiles.count():,} rows')

---

## 2. Logistic Regression — Evaluation <a id='2-logistic-regression--evaluation'></a>

Evaluate the supervised model that classifies customers as High Value
(`high_value = 1`) vs. Low Value (`high_value = 0`).

In [ ]:
# =============================================================================
# CONFUSION MATRIX
# =============================================================================
confusion = (
    lr_predictions
    .groupBy('high_value', 'prediction')
    .count()
    .orderBy('high_value', 'prediction')
)

print('Confusion Matrix (Spark):')
confusion.show()

# Convert to pandas for visualization
conf_pd = confusion.toPandas()
print('\nConfusion Matrix (detail):')
print(conf_pd.to_string(index=False))

In [ ]:
# =============================================================================
# CLASSIFICATION METRICS
# =============================================================================
# AUC-ROC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='high_value',
    rawPredictionCol='prediction',
    metricName='areaUnderROC'
)
auc_roc = auc_evaluator.evaluate(lr_predictions)

# Accuracy
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='high_value',
    predictionCol='prediction',
    metricName='accuracy'
)
accuracy = acc_evaluator.evaluate(lr_predictions)

# Precision
prec_evaluator = MulticlassClassificationEvaluator(
    labelCol='high_value',
    predictionCol='prediction',
    metricName='weightedPrecision'
)
precision = prec_evaluator.evaluate(lr_predictions)

# Recall
rec_evaluator = MulticlassClassificationEvaluator(
    labelCol='high_value',
    predictionCol='prediction',
    metricName='weightedRecall'
)
recall = rec_evaluator.evaluate(lr_predictions)

# F1
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol='high_value',
    predictionCol='prediction',
    metricName='f1'
)
f1 = f1_evaluator.evaluate(lr_predictions)

print('Logistic Regression — Classification Metrics')
print('=' * 45)
print(f'  AUC-ROC    : {auc_roc:.4f}')
print(f'  Accuracy   : {accuracy:.4f}')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1 Score   : {f1:.4f}')
print()

# Success metric validation
target_auc = 0.75
status = '✅ PASS' if auc_roc >= target_auc else '❌ FAIL'
print(f'Target AUC-ROC ≥ {target_auc}: {status} (actual: {auc_roc:.4f})')

In [ ]:
# =============================================================================
# VISUALIZATION: Confusion Matrix Heatmap
# =============================================================================
conf_pd = (
    lr_predictions
    .groupBy('high_value', 'prediction')
    .count()
    .toPandas()
)

# Pivot for heatmap
conf_matrix = conf_pd.pivot(
    index='high_value', columns='prediction', values='count'
).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    conf_matrix, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Low Value (0)', 'High Value (1)'],
    yticklabels=['Low Value (0)', 'High Value (1)'],
    ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.savefig(str(REPORTS_FIGURES / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORTS_FIGURES / "confusion_matrix.png"}')

---

## 3. K-Means — Cluster Interpretation <a id='3-k-means--cluster-interpretation'></a>

Interpret the 4 customer segments identified by K-Means (K=3 optimal by
Silhouette Score = 0.845, but K=4 was used in the pipeline). Each cluster
gets a business-meaningful name and actionable recommendation.

In [ ]:
# =============================================================================
# CLUSTER PROFILES — Business interpretation
# =============================================================================
print('K-Means Cluster Profiles:')
cluster_profiles.show(truncate=False)

### Cluster Business Interpretation

| Cluster | Name | Size | Key Characteristics | Marketing Action |
|---|---|---|---|---|
| 0 | **One-time Satisfied** | 72,013 (77%) | 1 order, low ticket ($112), high review (4.6), no repeat | Reactivation campaign — personalized offers to drive 2nd purchase |
| 1 | **Loyal Repeaters** | 703 (0.8%) | 2.3 orders, 256-day lifespan, moderate ticket ($128) | VIP program — reward loyalty, upsell premium products |
| 2 | **Multi-item Buyers** | 2,769 (3%) | 1.8 orders, 3.4 items/order, short lifespan (22 days) | Cross-sell bundles — they buy multiple items, optimize bundle offers |
| 3 | **High-ticket Critics** | 17,911 (19%) | 1 order, high ticket ($290), low review (2.1) | Service recovery — address dissatisfaction before churn, follow-up survey |

### Key Insights

1. **77% of customers are one-time buyers** — the biggest opportunity is converting
   them to repeat customers. Even a 5% conversion would add ~3,600 repeat customers.

2. **Loyal Repeaters are rare (0.8%)** but they represent the ideal customer profile.
   Understanding what drives their loyalty is key to scaling this segment.

3. **High-ticket Critics (19%) are a risk** — they spend the most per order but are
   dissatisfied (avg review 2.1). Service recovery here has the highest ROI because
   their ticket size is 2.6x the average.

4. **Multi-item Buyers (3%) buy fast and in bulk** — 3.4 items per order in a 22-day
   window suggests project-based purchasing (e.g., furnishing a room). Bundle
   recommendations are the natural strategy.

In [ ]:
# =============================================================================
# SILHOUETTE SCORE VALIDATION
# =============================================================================
# Reload customer features for silhouette calculation
customer_features = spark.read.parquet(str(DATA_PROCESSED / 'customer_features.parquet'))

# Need to rebuild features vector for evaluation
from pyspark.ml.feature import VectorAssembler

feature_cols = ['order_count', 'total_spent', 'avg_ticket',
                'items_purchased', 'avg_review_score']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
features_df = assembler.transform(customer_features)

# Join with cluster assignments
km_with_features = (
    features_df
    .join(
        km_clusters.select('customer_unique_id', 'cluster'),
        on='customer_unique_id',
        how='inner'
    )
)

# Calculate silhouette
sil_evaluator = ClusteringEvaluator(
    predictionCol='cluster',
    featuresCol='features',
    metricName='silhouette'
)
silhouette = sil_evaluator.evaluate(km_with_features)

target_sil = 0.40
status = '✅ PASS' if silhouette >= target_sil else '❌ FAIL'
print(f'K-Means Silhouette Score: {silhouette:.4f}')
print(f'Target Silhouette ≥ {target_sil}: {status}')

---

## 4. Cluster Visualization <a id='4-cluster-visualization'></a>

In [ ]:
# =============================================================================
# VISUALIZATION: Cluster Distribution
# =============================================================================
cluster_dist = (
    km_clusters
    .groupBy('cluster')
    .count()
    .orderBy('cluster')
    .toPandas()
)

cluster_names = {
    0: 'One-time\nSatisfied',
    1: 'Loyal\nRepeaters',
    2: 'Multi-item\nBuyers',
    3: 'High-ticket\nCritics'
}
cluster_dist['name'] = cluster_dist['cluster'].map(cluster_names)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cluster_dist['name'], cluster_dist['count'], color=sns.color_palette('Blues_d', 4))

for bar, count in zip(bars, cluster_dist['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{count:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('RetailMax — Customer Segments (K-Means, K=4)', fontsize=14)
ax.set_ylabel('Number of Customers')
ax.set_xlabel('Segment')
plt.tight_layout()
plt.savefig(str(REPORTS_FIGURES / 'cluster_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORTS_FIGURES / "cluster_distribution.png"}')

In [ ]:
# =============================================================================
# VISUALIZATION: Cluster Profiles — Grouped Bar Chart
# =============================================================================
profiles_pd = cluster_profiles.toPandas()

# Normalize metrics for comparison (min-max within each metric)
metrics = ['avg_orders', 'avg_spent', 'avg_ticket', 'avg_items', 'avg_review']
profiles_norm = profiles_pd.copy()
for m in metrics:
    col = profiles_norm[m].astype(float)
    mn, mx = col.min(), col.max()
    profiles_norm[m] = (col - mn) / (mx - mn) if mx > mn else 0

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics))
width = 0.2
names = ['One-time Satisfied', 'Loyal Repeaters', 'Multi-item Buyers', 'High-ticket Critics']
colors = sns.color_palette('Blues_d', 4)

for i, (_, row) in enumerate(profiles_norm.iterrows()):
    vals = [float(row[m]) for m in metrics]
    ax.bar(x + i * width, vals, width, label=names[i], color=colors[i])

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['Avg Orders', 'Avg Spent', 'Avg Ticket', 'Avg Items', 'Avg Review'])
ax.set_ylabel('Normalized Value (0–1)')
ax.set_title('RetailMax — Customer Segment Profiles (Normalized)', fontsize=14)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(str(REPORTS_FIGURES / 'cluster_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {REPORTS_FIGURES / "cluster_profiles.png"}')

---

## 5. Success Metrics Validation <a id='5-success-metrics-validation'></a>

In [ ]:
# =============================================================================
# SUCCESS METRICS — Problem Statement Canvas Validation
# =============================================================================
print('Problem Statement Canvas — Success Metrics Validation')
print('=' * 70)
print(f'{"Metric":40s} {"Target":>10s} {"Actual":>10s} {"Status":>8s}')
print('-' * 70)
print(f'{"AUC-ROC (Logistic Regression)":40s} {"≥ 0.75":>10s} {auc_roc:>10.4f} {"✅ PASS" if auc_roc >= 0.75 else "❌ FAIL":>8s}')
print(f'{"Silhouette Score (K-Means)":40s} {"≥ 0.40":>10s} {silhouette:>10.4f} {"✅ PASS" if silhouette >= 0.40 else "❌ FAIL":>8s}')

# Pipeline scale check
total_records = lr_predictions.count() + km_clusters.count()
scale_pass = total_records > 100_000
print(f'{"Pipeline processed > 100K records":40s} {"100K+":>10s} {total_records:>10,} {"✅ PASS" if scale_pass else "❌ FAIL":>8s}')

print('-' * 70)
all_pass = (auc_roc >= 0.75) and (silhouette >= 0.40) and scale_pass
print(f'Overall status: {"✅ ALL METRICS PASSED" if all_pass else "⚠️ REVIEW NEEDED"}')

---

## 6. Marketing Insights Report <a id='6-marketing-insights-report'></a>

### Strategic Recommendations for RetailMax Marketing Team

| Priority | Segment | Action | Expected Impact | KPI to Track |
|---|---|---|---|---|
| 🔴 HIGH | **High-ticket Critics** (19%) | Launch service recovery program — personalized follow-up within 48h of low-rated orders. Address delivery and product quality issues. | Reduce dissatisfaction from avg 2.1 to 3.5+ stars. Prevent churn of highest-spend segment ($290 avg ticket). | Review score improvement, repeat purchase rate |
| 🔴 HIGH | **One-time Satisfied** (77%) | Reactivation campaign — send personalized 2nd-purchase offer 30 days after first order. Use category of first purchase for recommendation. | Convert 5% to repeat → ~3,600 new repeat customers. | Reactivation rate, time to 2nd purchase |
| 🟡 MEDIUM | **Multi-item Buyers** (3%) | Design product bundles based on co-purchase patterns. Offer volume discounts on 3+ item orders. | Increase avg items from 3.4 to 4+ and basket size from $370 to $450. | Items per order, basket value |
| 🟢 LOW | **Loyal Repeaters** (0.8%) | Create VIP loyalty program — exclusive early access, free shipping, birthday rewards. Study their journey to replicate it. | Retain 95%+ of this segment. Use as model for expanding loyalty base. | Retention rate, NPS, referral rate |

### Business Impact Estimation

| Scenario | Current State | Target | Estimated Impact |
|---|---|---|---|
| Reactivate One-time Satisfied (5%) | 72,013 single-purchase | 3,600 repeat | ~$400K additional revenue (3,600 × $112 avg ticket) |
| Recover High-ticket Critics | 17,911 dissatisfied, $290 avg | Improve review 2.1→3.5 | Prevent ~$5.2M in churn risk (17,911 × $290) |
| Expand Loyal Repeaters | 703 customers (0.8%) | Double to 1.5% | ~$200K incremental from increased purchase frequency |

> **Methodology:** Revenue estimates based on segment avg_ticket × projected conversion.
> Assumptions: 5% reactivation rate is conservative (industry avg 10-15%);
> churn prevention assumes 50% of dissatisfied critics would not return.

---

## 7. LEAN Filter — Waste Elimination Review <a id='7-lean-filter--waste-elimination-review'></a>

| LEAN Question | Answer | Action |
|---|---|---|
| Does every metric link to a business decision? | ✅ Each metric validates a success criterion from the Problem Statement Canvas | Proceed |
| Are visualizations necessary? | ✅ Confusion matrix and cluster profiles support the marketing report narrative | Proceed |
| Is there redundancy with NB 04? | ✅ No — NB 04 trained models, NB 05 evaluates and interprets for business | Proceed |
| Is the marketing report actionable? | ✅ Each segment has a specific action, expected impact, and KPI to track | Proceed |

---

## 8. Decisions Log — Lesson 05 <a id='8-decisions-log--lesson-05'></a>

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| D1 | Use MulticlassClassificationEvaluator for precision/recall/F1 | Standard MLlib evaluator; provides weighted metrics for imbalanced classes | Manual calculation from confusion matrix (error-prone) | ✅ |
| D2 | Name clusters with business-meaningful labels | Marketing team needs actionable segment names, not "Cluster 0" | Technical labels only (not actionable) | ✅ |
| D3 | Include Business Impact Estimation with dollar values | Converts academic exercise into business case — critical for portfolio credibility | Metrics only without business translation (less impactful) | ✅ |
| D4 | Validate all success metrics from Problem Statement Canvas | Closes the CRISP-DM loop — Phase 5 must verify Phase 1 objectives | Skip validation (incomplete project) | ✅ |

---

## 9. Next Steps — Lesson 06 Preview <a id='9-next-steps--lesson-06-preview'></a>

| Priority | Next Step | Target |
|---|---|---|
| 🔴 Immediate | Generate final PDF report with results, metrics, and insights | NB 06 |
| 🔴 Immediate | Complete consigna compliance checklist — verify all requirements met | NB 06 |
| 🔴 Immediate | Write project retrospective (CRISP-DM cycle complete) | NB 06 |
| 🟡 Short-term | Complete README with full project documentation | After NB 06 |
| 🟢 Long-term | Validate pipeline on cloud (Colab/Databricks) for portfolio enhancement | Post-delivery |

---

**← Previous:** [04 — Modeling](./04_modeling.ipynb)
**Next Phase →** [06 — Deployment](./06_deployment.ipynb)